# Infrastructure as Code {background-color="black" background-image="images/servers-iac.png" background-size="100%"}


## Back to Swarm lesson

::::{.columns}

::: {.fragment .column width="50%"}

**What we did**

```bash
# 1. Launch VMs
multipass launch --name manager ...
multipass launch --name worker1 ...
multipass launch --name worker2 ...

# 2. Install Docker on each (or using cloud-init)
multipass exec worker1 -- \
  "curl -fsSL https://get.docker.com | sh"

# 3. Init + join the swarm
docker swarm init
JOIN=$(docker swarm join-token -q worker)
multipass exec worker1 -- \
  docker swarm join --token $JOIN ...
```

:::

::: {.fragment .column width="50%"}

**Problems**

- Not reproducible: "it worked on my machine"
- Not documented: future-you has no idea what you typed
- Error-prone: typos, wrong IPs, forgotten steps
- Not shareable: copy-paste from a chat (or slides) is not infrastructure specification
:::

::::


## What is Infrastructure as Code?

::::{.columns}

::: {.fragment .column width="50%"}

> *"Infrastructure as code (IaC) is the ability to provision and support your computing infrastructure **using code instead of manual processes and settings**."*
> [AWS](https://aws.amazon.com/what-is/iac/)

> *"IaC is the managing and provisioning of infrastructure through code instead of manual processes. Configuration files contain your infrastructure specifications, which makes it easier to **edit, distribute, and reproduce** configurations."*
> [Red Hat](https://www.redhat.com/en/topics/automation/what-is-infrastructure-as-code-iac)

:::

::: {.fragment .column width="50%"}

**Key aspects**

- Version-controlled like any other source code 
- Describes *desired state*: the tool figures out *how* to get there
- Generates the **same environment every time**: eliminates undocumented, ad hoc changes
- Integrates into **CI/CD pipelines**: infrastructure and application code deploy together

:::

::::

::: {.callout-tip .fragment}

**Red Hat best practice**: Immutable infrastructure: instead of patching existing servers, provision new ones with updated configuration and retire the old. Reduces drift and simplifies rollbacks.

:::


## Declarative vs. Imperative

<https://www.redhat.com/en/topics/automation/what-is-infrastructure-as-code-iac#declarative-vs-imperative>

::::{.columns}

::: {.fragment .column width="50%"}

**Declarative**

> A declarative approach to IaC defines the desired state of the system, including the resources you need and any properties they should have, and an IaC tool will configure it for you

- You say **what** you want
- Tool determines **how** to achieve it
- Re-running is safe: only differences applied (this is GOLD)
- **Examples:** Terraform, AWS CloudFormation

:::

::: {.fragment .column width="50%"}

**Imperative**

> An imperative approach of IaC defines the specific commands needed to achieve the desired configuration. Those commands then need to be executed in the correct order.

- You say **how** to get there, step by step
- Re-running may cause duplicate/conflicting actions
- Necessary when **order of events is critical**
- **Examples:** shell scripts, Python with Boto3/SDK

:::

::::

::: {.callout-note .fragment}
Most modern IaC tools use **declarative**. Spoilers: Ansible and K8S primarily uses a declarative YAML syntax.
:::


## Benefits of IaC

::::{.columns}

::: {.fragment .column width="33%"}

**Speed & consistency**

- Set up complete environments in minutes instead of hours or days
- Same IaC deploys identical environments for dev, test, and prod: no "works on my machine"
- Duplicate an entire environment in a different region (potentially cloud provider)

:::

::: {.fragment .column width="33%"}

**Reliability & safety**

- Reduces errors: manual configuration is error-prone by nature
- Mitigages configuration drift: environments stop diverging silently (jointly with Configuration Management tools)
- Roll back: to a previous stable state by reverting the code commit

:::

::: {.fragment .column width="33%"}

**Collaboration & governance**

- Reduces cost: less manual labour, fewer incidents (initial learning needed)
- Infrastructure can reviewed via pull requests, just like application code
- Integrates with CI/CD pipelines: infrastructure changes ship alongside app changes
- Enables security as code: firewall rules, IAM, encryption committed and audited

:::

::::


# Terraform{background-color="white" background-image="images/terraform.avif" background-size="100%"}


## What is Terraform
::::{.columns}

::: {.fragment .column width="50%"}

<https://www.terraform.io/>

- Open-source IaC tool by HashiCorp ([BSL](https://en.wikipedia.org/wiki/Business_Source_License) license; [OpenTofu](https://opentofu.org/) is the FOSS fork
- Single binary, runs anywhere — laptop, CI/CD, cloud shell
- Manages resources across **hundreds of providers**: AWS, GCP, Azure, Docker, Multipass, GitHub, …
- Tracks what it created in a **state file** → knows what to add, change, or destroy

:::

::: {.fragment .column width="50%"}

![](https://www.datocms-assets.com/2885/1620155116-brandhcterraformverticalcolor.svg)

:::

::::


## OpenTofu — The FOSS Fork

<https://opentofu.org/>

::::{.columns}

::: {.fragment .column width="50%"}

**The story**

- August 2023: HashiCorp relicenses Terraform from **MPL 2.0 → BSL 1.1**
- BSL = Business Source License: *not* OSI-approved open source
- Restriction: you cannot use Terraform to build a **competing product**
- September 2023: the community forks it → **OpenTofu**, under the **Linux Foundation**
- Today: OpenTofu 1.x — actively maintained, adding features ahead of Terraform

:::

::: {.fragment .column width="50%"}

**Why it matters for this course**

- Drop-in replacement: same HCL syntax, same providers, same workflow
- The only change: `terraform` → `tofu` on the CLI
- **We will use OpenTofu** for all hands-on work
- You'll encounter Terraform at work — the concepts are identical

```bash
# Everything you learn with tofu applies directly to terraform
tofu init   ≡   terraform init
tofu plan   ≡   terraform plan
tofu apply  ≡   terraform apply
```

:::

::::

::: {.callout-note .fragment}
IBM acquired HashiCorp in 2024. OpenTofu is the community's answer to ensuring the tool stays open.
:::


### Let's try

::::{.columns}

::: {.fragment .column width="50%"}
[Install](https://opentofu.org/docs/v1.11/intro/install/>)

```bash
nics@lics:~$ tofu --version
OpenTofu v1.11.5
on linux_amd64
   
```
:::

::: {.fragment .column width="50%"}

**Hello, World!**

```hcl
# main.tf
output "hello" {
  value = "Hello, World!"
}
```

```bash
$ tofu init && tofu apply -auto-approve
```

:::
::::

## HCL: HashiCorp Configuration Language

> OpenTofu language is defined in terms of a syntax called HCL...
<https://opentofu.org/docs/language/syntax/configuration/>

::::{.columns}

::: {.fragment .column width="50%"}

**Syntax**

```hcl
<BLOCK TYPE> "<BLOCK LABEL>" "<BLOCK LABEL>" {
  # Block body
  <IDENTIFIER> = <EXPRESSION> # Argument
}
```
- **Blocks** — containers representing objects (resources, providers…); have a type, optional labels, and a body
- **Arguments** — key = value pairs inside a block
- **Expressions** — values: literals, references, or computed combinations
- Let's give a look to [providers](https://search.opentofu.org/providers)
:::

::: {.fragment .column width="50%"}

**Example**

```hcl
terraform {
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 1.0.4"
    }
  }
}

variable "aws_region" {}

variable "base_cidr_block" {
  description = "A /16 CIDR range definition, such as 10.1.0.0/16, that the VPC will use"
  default = "10.1.0.0/16"
}

variable "availability_zones" {
  description = "A list of availability zones in which to create subnets"
  type = list(string)
}

provider "aws" {
  region = var.aws_region
}

resource "aws_vpc" "main" {
  # Referencing the base_cidr_block variable allows the network address
  # to be changed without modifying the configuration.
  cidr_block = var.base_cidr_block
}

resource "aws_subnet" "az" {
  # Create one subnet for each given availability zone.
  count = length(var.availability_zones)

  # For each subnet, use one of the specified availability zones.
  availability_zone = var.availability_zones[count.index]

  # By referencing the aws_vpc.main object, OpenTofu knows that the subnet
  # must be created only after the VPC is created.
  vpc_id = aws_vpc.main.id

  # Built-in functions and operators can be used for simple transformations of
  # values, such as computing a subnet address. Here we create a /20 prefix for
  # each subnet, using consecutive addresses for each availability zone,
  # such as 10.1.16.0/20 .
  cidr_block = cidrsubnet(aws_vpc.main.cidr_block, 4, count.index+1)
}
```
:::

::::



## The Terraform Workflow

```{mermaid}
flowchart LR
  W([✏️ WRITE]):::write --> I([⚙️ INIT]):::init
  I --> P([🔍 PLAN]):::plan
  P --> A([🚀 APPLY]):::apply
  A -.->|iterate| W
  A --> D([💣 DESTROY]):::destroy

  classDef write   fill:#4A90D9,stroke:#2c5f8a,color:#fff,rx:8
  classDef init    fill:#7B68EE,stroke:#4b3fa0,color:#fff
  classDef plan    fill:#F5A623,stroke:#c47d0e,color:#fff
  classDef apply   fill:#27AE60,stroke:#1a7a42,color:#fff
  classDef destroy fill:#E74C3C,stroke:#a93226,color:#fff
```

::::{.columns}

::: {.fragment .column width="25%"}
**init**

Downloads provider plugins defined in `required_providers`.  
Run once per project (or after adding a new provider).
:::

::: {.fragment .column width="25%"}
**plan**

Compares desired state (`.tf` files) with current state (`terraform.tfstate`).  
Shows exactly what will be created / changed / destroyed.
:::

::: {.fragment .column width="25%"}
**apply**

Executes the plan.  
Updates the state file.  
Safe to re-run — only applies *differences*.
:::

::: {.fragment .column width="25%"}
**destroy**
Deletes every resource tracked in state.  
:::

::::


## The State File

::::{.columns}

::: {.fragment .column width="50%"}

- Terraform writes `terraform.tfstate` after every `apply`
- Maps each HCL resource to its real-world counterpart (VM id, IP, …)
- **Never edit by hand** — use `tofu state` sub-commands
- **Local** (default): demos or single developer
- **Remote** (production): S3 + DynamoDB lock, Terraform Cloud, GitLab-managed state

:::

::: {.fragment .column width="50%"}

```
# terraform.tfstate (excerpt)
{
  "resources": [{
    "type": "multipass_instance",
    "name": "manager",
    "instances": [{
      "attributes": {
        "name": "manager",
        "ipv4": ["192.168.64.10"]
      }
    }]
  }]
}
```

:::

::::

:::{.callout-note .fragment}
Add `terraform.tfstate*` to `.gitignore`, it may contain secrets.
:::

# Hands-on — iaCACCA

Infrastracture as a Code for **C**lassroom **A**ttendance **C**ounter **C**omponent **A**pplication using Multipass and OpenTofu

## Project Layout

```
cacca/
└── terraform/
    ├── main.tf          ← provider + VM resources
    ├── variables.tf     ← knobs (image, CPUs, RAM …)
    ├── swarm.tf         ← null_resource provisioners for Swarm
    ├── outputs.tf       ← handy shortcuts printed after apply
    └── cloud-init.yaml  ← installs Docker at first boot
```

- Each `.tf` file is merged by Terraform at runtime, split it's a best practice, not a requirement
- `cloud-init.yaml` is referenced from `main.tf` via `file()`


## main.tf — Provider + VMs

::::{.columns}

::: {.fragment .column width="50%"}

**Declare the provider**
```hcl
# main.tf
terraform {
  required_providers {
    multipass = {
      source  = "larstobi/multipass"
      version = "1.4.3"
    }
  }
}

provider "multipass" {}
```

Provider docs: <https://registry.terraform.io/providers/larstobi/multipass/latest>

:::

::: {.fragment .column width="50%"}

**Manager + workers**
```hcl
resource "multipass_instance" "manager" {
  name           = "manager"
  image          = var.ubuntu_image   # "24.04"
  cpus           = var.manager_cpus   # 1
  memory         = var.manager_memory # "1G"
  disk           = var.manager_disk   # "5G"
  cloudinit_file = "${path.module}/cloud-init.yaml"
}

resource "multipass_instance" "worker" {
  count          = var.worker_count   # 2
  name           = "worker${count.index + 1}"
  image          = var.ubuntu_image
  cpus           = var.worker_cpus
  memory         = var.worker_memory
  disk           = var.worker_disk
  cloudinit_file = "${path.module}/cloud-init.yaml"
}
```

:::

::::


## cloud-init.yaml — Docker at Boot

::::{.columns}

::: {.fragment .column width="50%"}

```yaml
#cloud-config
packages:
  - docker.io
runcmd:
  - usermod -aG docker ubuntu
  - systemctl enable docker
  - systemctl start docker
```

:::

::: {.fragment .column width="50%"}

- `packages` → installed by `apt` before `runcmd`
- `usermod -aG docker ubuntu` → no `sudo` needed inside the VM
- `systemctl enable` → survives reboots
- Terraform passes this via `cloudinit_file = path` → Multipass passes it to `cloud-init` at launch

:::

::::


## Init & Plan

::::{.columns}

::: {.fragment .column width="50%"}

```bash
cd cacca/terraform

# Download the larstobi/multipass provider plugin
tofu init
```

Expected output:
```
Initializing provider plugins...
- Installing larstobi/multipass v1.4.3 ...
- Installing hashicorp/null v3.x.x ...

OpenTofu has been successfully initialized!
```

:::

::: {.fragment .column width="50%"}

```bash
# Preview what will be created — nothing is changed yet
tofu plan
```

Expected output:
```
Plan: 3 to add, 0 to change, 0 to destroy.

  + multipass_instance.manager
  + multipass_instance.worker[0]
  + multipass_instance.worker[1]
```

> **Read the plan before applying** — "no surprises" is the IaC guarantee.

:::

::::


## Apply — VMs Appear

```bash
tofu apply          # prompts "Do you want to perform these actions?"
# or skip the prompt:
tofu apply -auto-approve
```

::::{.columns}

::: {.fragment .column width="50%"}

Terraform creates the VMs:
```
multipass_instance.manager: Creating...
multipass_instance.worker[0]: Creating...
multipass_instance.worker[1]: Creating...
multipass_instance.manager: Creation complete.
multipass_instance.worker[0]: Creation complete.
multipass_instance.worker[1]: Creation complete.

Apply complete! Resources: 3 added.
```

:::

::: {.fragment .column width="50%"}

Verify from the host:
```bash
multipass list
# Name         State   IPv4
# manager      Running 192.168.64.10
# worker1      Running 192.168.64.11
# worker2      Running 192.168.64.12

multipass exec manager -- docker version
# Docker is already running — cloud-init did its job
```

:::

::::


## swarm.tf — Bootstrap the Cluster

::::{.columns}

::: {.fragment .column width="50%"}

**null_resource** — run a local command after the VMs exist

```hcl
# swarm.tf
resource "null_resource" "swarm_init" {
  triggers = { manager_name = multipass_instance.manager.name }
  depends_on = [multipass_instance.manager]

  provisioner "local-exec" {
    command = <<-EOT
      # wait for Docker to be ready (cloud-init may still be running)
      until multipass exec manager -- docker info > /dev/null 2>&1; do
        sleep 3
      done
      MANAGER_IP=$(multipass exec manager -- hostname -I | awk '{print $1}')
      multipass exec manager -- docker swarm init --advertise-addr "$MANAGER_IP"
    EOT
  }
}
```

:::

::: {.fragment .column width="50%"}

```hcl
resource "null_resource" "swarm_join" {
  count = var.worker_count
  triggers = {
    worker_name  = multipass_instance.worker[count.index].name
    manager_name = multipass_instance.manager.name
  }
  depends_on = [null_resource.swarm_init, multipass_instance.worker]

  provisioner "local-exec" {
    command = <<-EOT
      WORKER="${multipass_instance.worker[count.index].name}"
      until multipass exec "$WORKER" -- docker info > /dev/null 2>&1; do
        sleep 3
      done
      TOKEN=$(multipass exec manager -- docker swarm join-token worker -q)
      MANAGER_IP=$(multipass exec manager -- hostname -I | awk '{print $1}')
      multipass exec "$WORKER" -- docker swarm join --token "$TOKEN" "$MANAGER_IP:2377"
    EOT
  }
}
```

- `null_resource` has no cloud state — it just runs commands
- `depends_on` enforces the init → join ordering
- The join token is read live from the manager — never stored in state

:::

::::


## outputs.tf — Handy Shortcuts

```hcl
# outputs.tf
output "manager_name" {
  value = multipass_instance.manager.name
}

output "worker_names" {
  value = [for w in multipass_instance.worker : w.name]
}

output "useful_commands" {
  value = {
    list_vms      = "multipass list"
    swarm_nodes   = "multipass exec ${multipass_instance.manager.name} -- docker node ls"
    shell_manager = "multipass shell ${multipass_instance.manager.name}"
    destroy       = "tofu destroy"
  }
}
```

After `tofu apply`:
```bash
tofu output
# useful_commands = {
#   list_vms      = "multipass list"
#   swarm_nodes   = "multipass exec manager -- docker node ls"
#   shell_manager = "multipass shell manager"
#   destroy       = "tofu destroy"
# }
```

::: {.callout-tip .fragment}
Use `tofu output -raw manager_name` to capture a value in a shell script:
```bash
MANAGER=$(tofu output -raw manager_name)
multipass exec "$MANAGER" -- docker node ls
```
:::


## Infrastructure ready

`tofu apply` does **three things** in sequence:

1. Create VMs (`multipass_instance`)
2. Init the Swarm on the manager (`null_resource.swarm_init`)
3. Join each worker (`null_resource.swarm_join`)


::: {.fragment}

**Verify the Swarm is ready**
```bash
multipass exec manager -- docker node ls
# ID          HOSTNAME  STATUS  AVAILABILITY  MANAGER STATUS
# abc123 *    manager   Ready   Active        Leader
# def456      worker1   Ready   Active
# ghi789      worker2   Ready   Active
```
:::



## Build images

**Build & push images to Docker Hub** 

```bash
export DOCKER_USER=tapunict   # ← your Docker Hub username
docker login

docker build -t $DOCKER_USER/cacca-api:latest      cacca/api
docker build -t $DOCKER_USER/cacca-frontend:latest cacca/frontend
docker push $DOCKER_USER/cacca-api:latest
docker push $DOCKER_USER/cacca-frontend:latest
```

See the images <https://hub.docker.com/repositories/tapunict>

:::{.callout-note}
As alternative you have to copy the code into allo machines and build the images
:::


## Deploy the stack

```bash
# Copy the compose file into the manager VM
multipass transfer cacca/docker-compose.frontend.yml manager:docker-compose.frontend.yml

# Check
multipass exec manager ls 


# Deploy — all nodes pull images from Docker Hub
MANAGER_IP=$(multipass info manager | grep IPv4 | awk '{print $2}') \
  multipass exec manager -- \
  sh -c "MANAGER_IP=$MANAGER_IP docker stack deploy --detach -c docker-compose.frontend.yml cacca"


# Check services are running
multipass exec manager -- docker stack services cacca


# Open the browser and simulate an entry
curl -X POST "http://10.146.100.232:8000/entry?room=aula1"
```